# Retriever Tuning: Three RAG Configurations Compared

**Document:** `sample-sdf-document.pdf` (Cytiva storage-conditions letter + two Certificates of Quality, 3 pages)
**Stack:** PyMuPDF + LlamaIndex + BGE-small embeddings + Gemini (generation and reranking)
**Author:** Bashar

The embedding comparison ended with an open problem. For the query *"What test methods were used for quality control?"* the test table on page 3 is split across two chunks, and with `top_k=2` no embedding model retrieved both halves, so every answer listed four of the six tests.

This notebook keeps the document, chunking, embedding model and LLM fixed and changes only the **retriever settings**, to see whether that problem can be solved at retrieval time:

- `top_k`: how many chunks are retrieved
- **similarity threshold**: whether weak matches are filtered out
- **reranker**: whether an LLM re-sorts the retrieved chunks and keeps only the best

| Config | top_k | Threshold | Reranker | Purpose |
|---|---|---|---|---|
| A | 5 | None | Off | Baseline |
| B | 8 | 0.47 | Off | Cast a wider net, filter weak matches |
| C | 8 | 0.47 | On (keep top 3) | Same as B, then let an LLM pick the best chunks |

B and C differ only in the reranker, so its effect can be isolated.

## Step 0: Install dependencies

In [ ]:
%pip install -q pymupdf llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface google-genai nest_asyncio

In [1]:
import nest_asyncio
nest_asyncio.apply()  # Colab already runs an event loop; LlamaIndex needs to nest inside it

## Step 1: Gemini API key

Add the key as a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar) and allow this notebook to access it. Keeping the key in a secret rather than in a cell means the notebook can be shared without exposing it.

In [2]:
import os

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

assert GOOGLE_API_KEY and GOOGLE_API_KEY.strip(), "No Gemini key found. Add a Colab secret named GOOGLE_API_KEY."
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("Gemini key loaded.")

Gemini key loaded.


## Step 2: Upload the PDF

In [3]:
import glob

candidates = glob.glob("*sdf-document*.pdf") or glob.glob("/content/*sdf-document*.pdf")
if not candidates:
    from google.colab import files
    print("Please select 'sample-sdf-document.pdf' to upload:")
    uploaded = files.upload()
    candidates = list(uploaded.keys())

pdf_path = candidates[0]
print("Using PDF:", pdf_path)

Using PDF: sample-sdf-document.pdf


## Step 3: Build the index (held constant for every configuration)

Same setup as the embedding comparison: one `Document` per page, 128-token chunks with 20 overlap, and **BGE-small**, the model chosen in that task, with its query prefix.

In [4]:
import time
import fitz  # PyMuPDF
from llama_index.core import Document, Settings, VectorStoreIndex, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

def load_pdf_with_pymupdf(path):
    # Parse a PDF into one LlamaIndex Document per page.
    doc = fitz.open(path)
    documents = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if not text.strip():
            continue
        documents.append(Document(
            text=text,
            metadata={"page_number": i + 1},
            excluded_embed_metadata_keys=["page_number"],
        ))
    doc.close()
    return documents

pdf_docs = load_pdf_with_pymupdf(pdf_path)
nodes = SentenceSplitter(chunk_size=128, chunk_overlap=20).get_nodes_from_documents(pdf_docs)
chunk_no = {n.node_id: i for i, n in enumerate(nodes)}  # stable chunk numbers for the logs below

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")
llm = GoogleGenAI(model="models/gemini-3.8-flash", temperature=0)
Settings.embed_model = embed_model
Settings.llm = llm

index = VectorStoreIndex(nodes)
synthesizer = get_response_synthesizer(llm=llm)
print(f"{len(pdf_docs)} pages -> {len(nodes)} chunks indexed with BGE-small")

3 pages -> 12 chunks indexed with BGE-small


## Step 4: Query and ground truth

The page 3 certificate has a table headed **Description / Test Method / Result** with six rows. A complete answer names all six. Coverage is counted by checking for each test's name, in the retrieved context and in the generated answer.

One thing makes this query harder than it looks: the PDF holds **two** certificates. Page 2 (a different product) also lists Autoclave and Gamma Irradiation, as release *limits*. So those two words can reach the LLM from the wrong certificate. To keep that visible, context coverage is counted **only from page 3 chunks**, and page 2 chunks are reported separately as off-target.

In [5]:
QUERY = "What test methods were used for quality control?"

TESTS = {
    "Autoclave": "autoclave",
    "Gamma Irradiation": "gamma",
    "Flow Rate Test": "flow rate",
    "Pressure Integrity": "pressure integrity",
    "Visual Inspection": "visual inspection",
    "Package Integrity": "package integrity",
}
TARGET_PAGE = 3

def tests_in(text):
    return [name for name, key in TESTS.items() if key in text.lower()]

def context_coverage(retrieved):
    # tests present in retrieved chunks that come from the page 3 certificate
    target_text = " ".join(n.get_content() for n in retrieved if n.metadata["page_number"] == TARGET_PAGE)
    return tests_in(target_text)

def describe(retrieved):
    for r, n in enumerate(retrieved, 1):
        print(f"  [{r}] score={n.score:.4f} chunk {chunk_no[n.node.node_id]:>2} page {n.metadata['page_number']} | "
              + n.get_content()[:95].replace("\n", " | "))

table_chunks = [i for i, n in enumerate(nodes) if n.metadata["page_number"] == TARGET_PAGE and tests_in(n.get_content())]
print("Chunks holding the page 3 test table:", table_chunks)
for i in table_chunks:
    print(f"  chunk {i}: {tests_in(nodes[i].get_content())}")

Chunks holding the page 3 test table: [9, 10]
  chunk 9: ['Autoclave', 'Gamma Irradiation']
  chunk 10: ['Flow Rate Test', 'Pressure Integrity', 'Visual Inspection', 'Package Integrity']


## Step 5: Look at the actual scores before choosing a threshold

The module's advice is to check the model's real score range first. This ranks all 12 chunks for the query.

In [6]:
full_ranking = index.as_retriever(similarity_top_k=len(nodes)).retrieve(QUERY)
describe(full_ranking)
scores = [n.score for n in full_ranking]
print(f"\nScore range for this query: {min(scores):.3f} to {max(scores):.3f}")

  [1] score=0.5804 chunk 10 page 3 | 0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min 
  [2] score=0.5336 chunk  8 page 3 | Cytiva | cytiva.com | Certificate of Quality | This product is manufactured in compliance with our IS
  [3] score=0.5274 chunk  6 page 2 | Modified, ÄKTATM ready | Expiration Date:  20230126  | Product Release Criteria  | We hereby certify 
  [4] score=0.5272 chunk  7 page 2 | Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40
  [5] score=0.5268 chunk  4 page 2 | Certificate of Quality  | This product is manufactured in compliance with our ISO 9001 certified 
  [6] score=0.4863 chunk  5 page 2 | cytiva.com  | 29190939 Rev AD | Product: ÄKTA™ ready Gradient Flow Section With Inlets | Lot Number: 
  [7] score=0.4802 chunk  9 page 3 | Product: | AKTA ready Low Flow Kit | Lot Number: | 18356721 | Product Article Number: | 28 9301 82 | Date o
  [8] score

## Step 6: Experiment 1, top_k on its own

In [7]:
import pandas as pd

rows = []
for k in [2, 5, 8, 10]:
    retrieved = index.as_retriever(similarity_top_k=k).retrieve(QUERY)
    cov = context_coverage(retrieved)
    rows.append({
        "top_k": k,
        "Tests in context (of 6)": len(cov),
        "Table chunks retrieved": sorted(chunk_no[n.node.node_id] for n in retrieved if chunk_no[n.node.node_id] in table_chunks),
        "Page 2 chunks (other certificate)": sum(n.metadata["page_number"] == 2 for n in retrieved),
        "Page 1 chunks (storage letter)": sum(n.metadata["page_number"] == 1 for n in retrieved),
        "Context words": sum(len(n.get_content().split()) for n in retrieved),
    })
pd.DataFrame(rows).set_index("top_k")

,Tests in context (of 6),Table chunks retrieved,Page 2 chunks (other certificate),Page 1 chunks (storage letter),Context words
top_k,,,,,
2,4,[10],0,0,93
5,4,[10],3,0,264
8,6,"[9, 10]",4,1,432
10,6,"[9, 10]",4,3,515


## Step 7: Experiment 2, similarity threshold on its own

Thresholds are applied to the top 10 chunks. The module's suggested values (0.10 to 0.20) are included alongside values taken from BGE's actual score range above.

In [8]:
top10 = index.as_retriever(similarity_top_k=10).retrieve(QUERY)

rows = []
for threshold in [0.15, 0.20, 0.45, 0.47, 0.50, 0.55]:
    kept = [n for n in top10 if n.score and n.score > threshold]
    cov = context_coverage(kept)
    rows.append({
        "Threshold": threshold,
        "Chunks kept (of 10)": len(kept),
        "Tests in context (of 6)": len(cov),
        "Table chunks kept": sorted(chunk_no[n.node.node_id] for n in kept if chunk_no[n.node.node_id] in table_chunks),
        "Page 2 chunks kept": sum(n.metadata["page_number"] == 2 for n in kept),
    })
pd.DataFrame(rows).set_index("Threshold")

,Chunks kept (of 10),Tests in context (of 6),Table chunks kept,Page 2 chunks kept
Threshold,,,,
0.15,10,6,"[9, 10]",4
0.20,10,6,"[9, 10]",4
0.45,10,6,"[9, 10]",4
0.47,7,6,"[9, 10]",4
0.50,5,4,[10],3
0.55,1,4,[10],0


## Step 8: The three configurations

The reranker is LlamaIndex's `LLMRerank` with Gemini: it shows the LLM the retrieved chunks, asks it to score each for relevance to the question, and keeps the top 3. Unlike embedding similarity, the LLM reads the query and the chunk together.

In [ ]:
from llama_index.core.postprocessor import LLMRerank
from llama_index.core.schema import QueryBundle

CONFIGS = [
    {"name": "A", "top_k": 5, "threshold": None, "rerank": False, "note": "Baseline"},
    {"name": "B", "top_k": 8, "threshold": 0.47, "rerank": False, "note": "Wider net + threshold"},
    {"name": "C", "top_k": 8, "threshold": 0.47, "rerank": True, "note": "B + LLM reranker (top 3)"},
]
RERANK_TOP_N = 3

def with_retries(fn, retries=5):
    # Gemini's free tier occasionally returns 503 "high demand"; wait and retry instead of failing the run
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"  Gemini call failed ({type(e).__name__}); retrying in {wait}s")
            time.sleep(wait)

def run_config(cfg):
    retrieved = index.as_retriever(similarity_top_k=cfg["top_k"]).retrieve(QUERY)
    n_retrieved = len(retrieved)

    if cfg["threshold"] is not None:
        retrieved = [n for n in retrieved if n.score and n.score > cfg["threshold"]]
    n_after_threshold = len(retrieved)

    if cfg["rerank"]:
        reranker = LLMRerank(llm=llm, choice_batch_size=10, top_n=RERANK_TOP_N)
        retrieved = with_retries(lambda: reranker.postprocess_nodes(retrieved, QueryBundle(QUERY)))

    answer = with_retries(lambda: str(synthesizer.synthesize(QUERY, nodes=retrieved)).strip())
    return {"cfg": cfg, "nodes": retrieved, "answer": answer,
            "n_retrieved": n_retrieved, "n_after_threshold": n_after_threshold}

runs = []
for cfg in CONFIGS:
    print(f"\n{'=' * 100}\nCONFIG {cfg['name']}: top_k={cfg['top_k']}, threshold={cfg['threshold']}, "
          f"reranker={'on' if cfg['rerank'] else 'off'}  ({cfg['note']})\n{'=' * 100}")
    run = run_config(cfg)
    runs.append(run)
    print(f"retrieved {run['n_retrieved']} -> after threshold {run['n_after_threshold']} -> sent to LLM {len(run['nodes'])}")
    if cfg["rerank"]:
        print("(scores below are the reranker's 1-10 relevance ratings, not embedding similarity)")
    print("\nRETRIEVED CHUNKS (short view):")
    describe(run["nodes"])
    print("\nRETRIEVED CHUNKS (full text):")
    for r, n in enumerate(run["nodes"], 1):
        print(f"\n--- Chunk {r} (score {n.score:.2f}, page {n.metadata['page_number']}) ---")
        print(n.get_content())
    print(f"\nFINAL ANSWER:\n{run['answer']}")

## Step 9: Side-by-side comparison

In [ ]:
rows = []
for run in runs:
    cfg, retrieved = run["cfg"], run["nodes"]
    ctx = context_coverage(retrieved)
    ans = tests_in(run["answer"])
    rows.append({
        "Config": cfg["name"],
        "top_k": cfg["top_k"],
        "Threshold": "None" if cfg["threshold"] is None else cfg["threshold"],
        "Reranker": "On" if cfg["rerank"] else "Off",
        "Chunks sent to LLM": len(retrieved),
        "Context words": sum(len(n.get_content().split()) for n in retrieved),
        "Page 3 tests in context": f"{len(ctx)}/6",
        "Tests named in answer": f"{len(ans)}/6",
        "Missing from answer": ", ".join(t for t in TESTS if t not in ans) or "none",
        "Off-target chunks (pages 1-2)": sum(n.metadata["page_number"] != TARGET_PAGE for n in retrieved),
        # EMA/410/01 appears only on the page 2 certificate, so it marks content leaking in from the other product
        "Other certificate's EMA/410/01 in answer": "Yes" if "EMA/410" in run["answer"] else "No",
    })
comparison = pd.DataFrame(rows).set_index("Config")
comparison

## Findings

Results from a run of this notebook (BGE-small, 12 chunks, Gemini at temperature 0).

| Config | top_k | Threshold | Reranker | Chunks sent to LLM | Context words | Tests named in answer | Off-target chunks | Other certificate's EMA/410/01 in answer |
|---|---|---|---|---|---|---|---|---|
| A | 5 | None | Off | 5 | 264 | 4 of 6 | 3 | No |
| B | 8 | 0.47 | Off | 7 | 371 | 6 of 6 | 4 | Yes |
| C | 8 | 0.47 | On (top 3) | 3 | 187 | 6 of 6 | 1 | No |

**1. The baseline misses a third of the answer.** With `top_k=5`, the retriever returned the second half of the test table (chunk 10) but not the first half (chunk 9), which holds the Autoclave and Gamma Irradiation rows. Embedding similarity ranks chunk 9 only 7th of 12, because most of that chunk is product and lot details and the table only starts in its last lines. Three of the five slots went to page 2, which is the certificate for a different product. The answer listed four of the six tests.

**2. Raising top_k to 8 fixes completeness but adds noise.** At `top_k=8` both halves of the table are retrieved and the answer names all six tests. The cost is a context of 7 chunks and 371 words, four of them from the other certificate. That noise reached the answer: Config B's answer says the materials are "compliant with EMA/410/01", which appears only on the page 2 certificate, not on the one the test table belongs to. The answer is complete but mixes two products.

**3. The threshold did very little, and the suggested values did nothing.** BGE's scores for this query run from 0.44 to 0.58, so the module's example thresholds of 0.10 to 0.20 keep every chunk. A threshold has to be set from the model's real score range. Even then it is a blunt tool here: 0.47 removes one chunk out of eight, while 0.50 removes chunk 9 (score 0.48) and takes the answer back to four of six tests. The chunk that is needed scores *lower* than five chunks that are not, so no threshold can keep it and drop them.

**4. The reranker is what made the difference.** Given the same 7 chunks as Config B, the LLM reranker rated the two table chunks 9 and 8 out of 10 and moved chunk 9 from 7th place to 2nd. It kept 3 chunks instead of 7, cut the context roughly in half (187 words), and the answer still named all six tests, this time without the EMA/410/01 detail from the other certificate. It reads the question and the chunk together, so it recognised the table header "Description / Test Method / Result" as relevant where the embedding score did not.

**5. The reranker only helps if the right chunk is in the pool.** The module's example for Config C uses `top_k=5` with reranking. On this document that would not work: chunk 9 is ranked 7th, so it is never retrieved at `top_k=5` and there is nothing for the reranker to promote. Reranking reorders what retrieval found; it cannot add what retrieval missed. A wider first-stage `top_k` followed by a reranker is the combination that works.

**Best configuration: C** (`top_k=8`, threshold 0.47, LLM reranker keeping the top 3). It gave the complete answer from the smallest and cleanest context.

**Trade-offs and limits.**
- Config C makes one extra LLM call per query for reranking, so it is slower and costs more than A or B. In the run above the free Gemini tier also returned temporary errors that needed retries.
- Its third chunk was still from the other certificate (rated 5 of 10). `top_n=2` would have removed it for this query, but that is tuning to a single question.
- The 0.47 threshold was chosen after looking at this query's scores. It should be checked against other queries before being relied on.
- The root cause is the chunking: a table split across two chunks. Retriever tuning works around it; larger or table-aware chunks would fix it at the source.
- One query on one 3-page document shows the mechanism, not a general rule.